IMPORTS

In [1]:
from typing import List, Union
from pathlib import Path

import string
import pandas as pd
import numpy as np
import itertools
import re

C:\Users\zscoman\AppData\Local\Temp\ipykernel_20908\1879127891.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## FILE READER

In [2]:
def file_reader(csv_path_list, ds_count:int=0, fl_count:int=0):
    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    org_tabs = []
    contact_tabs = []
    dist_tabs = []
    region_tabs = []
    
    org = "_organelles"
    contacts = "_contacts"
    dist = "_distributions"
    regions = "_regions"

    for loc in csv_path_list:
        if isinstance(loc, str): loc = Path(loc)
        ds_count = ds_count + 1
        files_store = sorted(loc.glob("*.csv"))
        for file in files_store:
            fl_count = fl_count + 1
            stem = file.stem

            if org in stem:
                test_orgs = pd.read_csv(file, index_col=0)
                test_orgs.insert(0, "dataset", stem[:-11])
                org_tabs.append(test_orgs)
            if contacts in stem:
                test_contact = pd.read_csv(file, index_col=0)
                test_contact.insert(0, "dataset", stem[:-9])
                contact_tabs.append(test_contact)
            if dist in stem:
                test_dist = pd.read_csv(file, index_col=0)
                test_dist.insert(0, "dataset", stem[:-14])
                dist_tabs.append(test_dist)
            if regions in stem:
                test_regions = pd.read_csv(file, index_col=0)
                test_regions.insert(0, "dataset", stem[:-8])
                region_tabs.append(test_regions)
            
    org_df = pd.concat(org_tabs,axis=0, join='outer')
    contacts_df = pd.concat(contact_tabs,axis=0, join='outer')
    dist_df = pd.concat(dist_tabs,axis=0, join='outer')
    regions_df = pd.concat(region_tabs,axis=0, join='outer')
    return org_df, contacts_df, dist_df, regions_df, ds_count, fl_count


## Orgs

In [3]:
def org_summarize(org_df,
                  multi_regions: list[bool],
                  group_by:list[str]=['dataset', 'image_name', 'cell_number', 'object'], 
                  shared_columns:list[str]=["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"],
                  ag_func_standard: list[str]=['mean', 'median', 'std'],
                  zeros=None):
    #################
    # Basic Summaries
    #################
    for col in ['cell_col', 'subregion']:
        tab1 = org_df[group_by + [col,'volume']].groupby(group_by+[col]).agg(['count', 'sum'] + ag_func_standard)
        tab2 = org_df[group_by + [col,'surface_area']].groupby(group_by+[col]).agg(['sum'] + ag_func_standard)
        tab3 = org_df[group_by + [col]+shared_columns].groupby(group_by+[col]).agg(ag_func_standard)
        shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by+[col])
        shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by+[col]).unstack(col).swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
        if col == 'cell_col':
            org_summary = shared_metrics
        else:
            org_summary = pd.merge(org_summary, shared_metrics, 'outer', on=group_by)
    # #################
    # # Regions w/ Multiple Instances
    # #################
    tab1 = org_df[multi_regions][(group_by + ['subregion_separated','volume'])].groupby(group_by+['subregion_separated']).agg(['count', 'sum'] + ag_func_standard)
    tab2 = org_df[multi_regions][(group_by + ['subregion_separated','surface_area'])].groupby(group_by+['subregion_separated']).agg(['sum'] + ag_func_standard)
    tab3 = org_df[multi_regions][(group_by + ['subregion_separated']+shared_columns)].groupby(group_by+['subregion_separated']).agg(ag_func_standard)
    shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by+['subregion_separated'])
    shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by+['subregion_separated']).unstack('subregion_separated').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    org_summary = pd.merge(org_summary, shared_metrics, 'outer', on=group_by)
    org_summary = org_summary.reindex(org_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    for fn in ag_func_standard:
        for region in org_summary.columns.get_level_values(0).unique():
            count_one = [count!=1 for count in org_summary[region]["volume"]["count"]]
            org_summary.loc[count_one, ([region],["equivalent_diameter"],[fn])] = zeros
    return org_summary

In [4]:
def summarize_org(org_df, group_by, ag_func_standard):
    columns2 = [col for col in org_df.columns if col.endswith(("_count", "_volume"))]
    contact_counts_summary = org_df[group_by + columns2].groupby(group_by).agg(['sum'] + ag_func_standard)
    org_summary = pd.merge(org_summary, contact_counts_summary, 'outer', on=group_by)#left_on=group_by, right_on=True)
    return org_summary

In [5]:
def org_area_fraction(org_summary, regions_summary, regions_df, group_by): # Impacted by separating functions
    regions_df = regions_df.set_index(group_by + ['label'])
    for region in org_summary.columns.get_level_values(0).unique():
        area_fraction=[]
        if '-' in region:   #specific cell multi region, must use regions_df
            for idx in org_summary.index.unique():
                org_vol = org_summary.loc[idx][(region, 'volume', 'sum')]
                cell_vol = sum([regions_df.loc[idx[:-1] + (reg.split('-')[0],int(reg.split('-')[1]))]["volume"] for reg in region.split(':')[-1].split("_")])
                afrac = org_vol/cell_vol
                area_fraction.append(afrac)
        else:               #summarized cell region, must use regions_summary
            for idx in org_summary.index.unique():
                org_vol = org_summary.loc[idx][(region, 'volume', 'sum')]
                cell_vol = sum([regions_summary.loc[idx[:-1] + (reg,)]["volume"]["sum"] for reg in region.split(':')[-1].split("_")])
                afrac = org_vol/cell_vol
                area_fraction.append(afrac)
        org_summary[(region, 'volume', 'fraction')] = area_fraction
    return org_summary

In [6]:
def normalize_org(org_df, org_summary, group_by, multi_regions):
    cont_cnt_subcell = org_df[group_by+['subregion']]
    cont_cnt_subcell[[col.split('_')[0] for col in org_df.columns if col.endswith(("_count"))]] = org_df[[col for col in org_df.columns if col.endswith(("_count"))]].astype(bool)
    cont_cnt_perorg_subcell = cont_cnt_subcell.groupby(group_by+['subregion']).agg('sum')
    cont_cnt_perorg_subcell.columns = pd.MultiIndex.from_product([cont_cnt_perorg_subcell.columns, ['count_in']])
    cont_cnt_perorg_subcell = cont_cnt_perorg_subcell.unstack('subregion').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    
    cont_cnt_cell = org_df[group_by+['cell_col']]
    cont_cnt_cell[[col.split('_')[0] for col in org_df.columns if col.endswith(("_count"))]] = org_df[[col for col in org_df.columns if col.endswith(("_count"))]].astype(bool)
    cont_cnt_perorg_cell = cont_cnt_cell.groupby(group_by+['cell_col']).agg('sum')
    cont_cnt_perorg_cell.columns = pd.MultiIndex.from_product([cont_cnt_perorg_cell.columns, ['count_in']])
    cont_cnt_perorg_cell = cont_cnt_perorg_cell.unstack('cell_col').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    
    cont_cnt_subcell_sep = org_df[multi_regions][group_by+['subregion_separated']]
    cont_cnt_subcell_sep[[col.split('_')[0] for col in org_df.columns if col.endswith(("_count"))]] = org_df[[col for col in org_df.columns if col.endswith(("_count"))]].astype(bool)
    cont_cnt_perorg_subcell_sep = cont_cnt_subcell_sep.groupby(group_by+['subregion_separated']).agg('sum')
    cont_cnt_perorg_subcell_sep.columns = pd.MultiIndex.from_product([cont_cnt_perorg_subcell_sep.columns, ['count_in']])
    cont_cnt_perorg_subcell_sep = cont_cnt_perorg_subcell_sep.unstack('subregion_separated').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    
    cont_cnt_perorg_perreg = pd.merge(cont_cnt_perorg_cell, cont_cnt_perorg_subcell, 'outer', on=group_by)
    cont_cnt_perorg_perreg = pd.merge(cont_cnt_perorg_perreg, cont_cnt_perorg_subcell_sep, 'outer', on=group_by)
    cont_cnt_perorg_perreg = cont_cnt_perorg_perreg.reindex(cont_cnt_perorg_perreg.columns.get_level_values(0).unique(), level=0, axis=1)
    
    for region in org_summary.columns.get_level_values(0).unique():
        for col in cont_cnt_perorg_perreg.columns:
            cont_cnt_perorg_perreg[(region, col[1], 'num_fraction_in')] = cont_cnt_perorg_perreg[col].values/org_summary[(region, 'volume', 'count')].values
    cont_cnt_perorg_perreg.sort_index(axis=1, inplace=True)
    org_summary = pd.merge(org_summary, cont_cnt_perorg_perreg, on=group_by, how='outer')
    return org_summary

In [7]:
def org_flattening(org_summary, org_df, splitter):
    all_combos = []
    all_orgs = list(set(org_df.loc[:, 'object'].tolist()))
    org_final = org_summary.unstack(-1)
    for col in org_final.columns:
        if col[2] in ('count_in', 'num_fraction_in') or col[1].endswith(('_count', '_volume')):
            if col[3] not in col[1]:
                org_final.drop(col,axis=1, inplace=True)
    
    org_final.columns = ["_".join((col_name[-1], col_name[2], col_name[1], col_name[0])) for col_name in org_final.columns.to_flat_index()]
    org_final = org_final.groupby(level=0, axis=1).sum()

    #renaming, filling "NaN" with 0 when needed, and removing ER_std columns
    for col in org_final.columns:
        if '_count_in_' or '_fraction_in_' in col:
            org_final[col] = org_final[col].fillna(0)

        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            org_final[col] = org_final[col].fillna(0)
            
        if col.endswith("_count_volume"):
            org_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)

        if col.startswith("ER_std_"):
            org_final.drop(columns=[col], inplace=True)
            
    return org_final

## Interactions

In [8]:
# TAKES IN:
#       INTERACTION DATA TABLE   -   For reorganizing the data
#       SPLITTER                 -   For determining orgs involved
def batch_summary_interactions(interaction_df, splitter):
    
    # CREATE A NEW DATA TABLE FOR INTERACTION COUNTS
    interaction_cnt = interaction_df[["dataset", "image_name", 'cell_number', "object", "label", "volume"]]
    
    # CREATES NEW COLUMNS EQUAL TO MAX NUMBER OF ORGANELLES INVOLVED IN A CONTACT
    # CREATES A NEW COLUMN FOR STORING THE ORGANELLE ID FOR EACH ORGANELLE INVOLVED IN A CONTACT
    # FOR EXAMPLE: mitoXER of 06_01 would become the following
    #              A_ID | orgA | B_ID | orgB
    #               06  | mito |  01  |  ER 
    interaction_cnt[[f"org{cha}" for cha in string.ascii_uppercase[:(len(max(interaction_cnt["object"].str.split(splitter), key=len)))]]] = interaction_cnt["object"].str.split(splitter, expand=True)
    interaction_cnt[[f"{cha}_ID" for cha in string.ascii_uppercase[:(len(max(interaction_cnt["label"].str.split('_'), key=len)))]]] = interaction_cnt["label"].str.split('_', expand=True)
    #iterating from a to val
    unstacked_interactions = []
    for cha in string.ascii_uppercase[:len(max(interaction_cnt["object"].str.split(splitter), key=len))]:
        # DETERMINE WHICH VALUES IN interaction_cnt HAVE THE DESIRED ALPHABETICAL ORGANELLE COUNT
        # i.e. if a contact has 4 organelles in it, the maximum alphabetical organelle count will be "D"
        valid = (interaction_cnt[f"org{cha}"] != None) & (interaction_cnt[f"{cha}_ID"] != None)

        # CREATES A NEW COLUMN WITH THE VALUE OF THE CURRENT ALPHABETICAL CHARACTER
        interaction_cnt[f"{cha}"] = None

        # Note: USED LATER WITH SEPARATING ORG AND INTERACTION DATA
        interaction_cnt.loc[valid, f"{cha}"] = interaction_cnt[f"org{cha}"] + "_" + interaction_cnt[f"{cha}_ID"]

        # CREATES A NEW DATAFRAME FOR PER CELL DATA FOCUSING ONLY ON CURRENT ALPHABETICAL ORGANELLE CHARACTER GROUPED BY CELL
        interaction_cnt_percell = interaction_cnt[["dataset", "image_name", 'cell_number', f"org{cha}", f"{cha}_ID", "object", "volume"]].groupby(["dataset", "image_name", 'cell_number', f"org{cha}", f"{cha}_ID", "object"]).agg(["count", "sum"])
        interaction_cnt_percell.columns = ["_".join(col_name).rstrip('_') for col_name in interaction_cnt_percell.columns.to_flat_index()]
        unstacked = interaction_cnt_percell.unstack(level='object')
        unstacked.columns = ["_".join(col_name).rstrip('_') for col_name in unstacked.columns.to_flat_index()]
        unstacked = unstacked.reset_index()
        for col in unstacked.columns:
            if col.startswith("volume_count_"):
                newname = col.split("_")[-1] + "_count"
                unstacked.rename(columns={col:newname}, inplace=True)
            if col.startswith("volume_sum_"):
                newname = col.split("_")[-1] + "_volume"
                unstacked.rename(columns={col:newname}, inplace=True)
        unstacked.rename(columns={f"org{cha}":"object", f"{cha}_ID":"label"}, inplace=True)
        unstacked.set_index(['dataset', 'image_name', 'cell_number', 'object', 'label'])    
        unstacked_interactions.append(unstacked)
    interaction_cnt = pd.concat(unstacked_interactions, axis=0).sort_index(axis=0)
    interaction_cnt = interaction_cnt.groupby(['dataset', 'image_name', 'cell_number', 'object', 'label']).sum().reset_index()                 #adds together all duplicates at the index, then resets the index
    interaction_cnt['label'] = interaction_cnt['label'].astype("Int64")  
    return interaction_cnt

In [9]:
def normalize_interaction_volumes(interaction_summary, 
                                  region_summary, 
                                  regions_df,
                                  group_by,
                                  splitter: str="X"):
    regions_df = regions_df.set_index(group_by + ['label'])
    for region in interaction_summary.columns.get_level_values(0).unique():
        norm_to_list = {}
        for idx,cha in enumerate(string.ascii_uppercase[:len(max(interaction_summary.index.get_level_values('object').str.split(splitter), key=len))]):
            for row in interaction_summary.index:
                if cha not in norm_to_list:
                    norm_to_list[cha]=[]
                if ((idx+1) <= len(row[-1].split(splitter))): # continue if nth order 
                    org = row[-1].split(splitter)[idx]
                    if '-' in region:
                        if (interaction_summary.loc[row][(region,'volume', 'sum')]>=0) and (sum([regions_df.loc[row[:-1] + (reg.split('-')[0],int(reg.split('-')[1]))]["volume"] for reg in region.split(':')[-1].split("_")])>=0):
                            norm_to_list[cha].append(interaction_summary.loc[row][(region,'volume', 'sum')]/sum([regions_df.loc[row[:-1] + (reg.split('-')[0],int(reg.split('-')[1]))]["volume"] for reg in region.split(':')[-1].split("_")]))
                        else:
                            norm_to_list[cha].append(None)
                    else:
                        if (interaction_summary.loc[row][(region,'volume', 'sum')]>=0) and (sum([region_summary.loc[row[:-1]+(reg.split('-')[0],)][(f"{org}_volume", 'sum')] for reg in region.split(':')[-1].split("_")]) >= 0):
                            norm_to_list[cha].append(interaction_summary.loc[row][(region,'volume', 'sum')]/sum([region_summary.loc[row[:-1]+(reg.split('-')[0],)][(f"{org}_volume", 'sum')] for reg in region.split(':')[-1].split("_")]))
                        else:
                            norm_to_list[cha].append(None)
                else: # specified interaction is below nth order, leave cell as none
                    norm_to_list[cha].append(None)
        for cha in string.ascii_uppercase[:len(max(interaction_summary.index.get_level_values('object').str.split(splitter), key=len))]:
            interaction_summary[(region, 'volume', f'norm_to_{cha}')] = norm_to_list[cha]
    interaction_summary = interaction_summary.reindex(interaction_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    return interaction_summary

In [10]:
def interaction_summarize(interaction_df,
                          multi_regions: list[bool],
                          group_by:list[str]=['dataset', 'image_name', 'object', 'cell_number'], 
                          shared_columns:list[str]=["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"],
                          ag_func_standard: list[str]=['mean', 'median', 'std']):   
    #################
    # Basic Summaries
    #################
    for col in ['cell_col', 'subregion']:
        tab1 = interaction_df[group_by + [col,'volume']].groupby(group_by+[col]).agg(['count', 'sum'] + ag_func_standard)
        tab2 = interaction_df[group_by + [col,'surface_area']].groupby(group_by+[col]).agg(['sum'] + ag_func_standard)
        tab3 = interaction_df[group_by + [col]+shared_columns].groupby(group_by+[col]).agg(ag_func_standard)
        shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by+[col])
        shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by+[col]).unstack(col).swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
        if col == 'cell_col':
            interaction_summary = shared_metrics
        else:
            interaction_summary = pd.merge(interaction_summary, shared_metrics, 'inner', on=group_by)

    #################
    # Regions w/ Multiple Instances
    #################
    tab1 = interaction_df.loc[multi_regions, (group_by + ['subregion_separated','volume'])].groupby(group_by+['subregion_separated']).agg(['count', 'sum'] + ag_func_standard)
    tab2 = interaction_df.loc[multi_regions, (group_by + ['subregion_separated','surface_area'])].groupby(group_by+['subregion_separated']).agg(['sum'] + ag_func_standard)
    tab3 = interaction_df.loc[multi_regions, (group_by + ['subregion_separated']+shared_columns)].groupby(group_by+['subregion_separated']).agg(ag_func_standard)
    shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by+['subregion_separated'])
    shared_metrics = pd.merge(shared_metrics, tab3, 'outer', on=group_by+['subregion_separated']).unstack('subregion_separated').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    interaction_summary = pd.merge(interaction_summary, shared_metrics, 'outer', on=group_by)
    interaction_summary = interaction_summary.reindex(interaction_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    
    return interaction_summary 

In [11]:
def interaction_flattening(interaction_summary):
    # contacts flattened
    interaction_final = interaction_summary.unstack(-1)
    interaction_final.columns = ["_".join((col_name[-1], col_name[2], col_name[1], col_name[0])) for col_name in interaction_final.columns.to_flat_index()]
    interaction_final = interaction_final.groupby(level=0, axis=1).sum()
    #renaming and filling "NaN" with 0 when needed
    for col in interaction_final.columns:
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            interaction_final[col] = interaction_final[col].fillna(0)
        if col.endswith("_count_volume"):
            interaction_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
    interaction_final = interaction_final.reset_index()
    return interaction_final

## Region

In [12]:
def region_volume_fraction(regions_summary):
    _vol = [s for s in regions_summary.columns.get_level_values(0).tolist() if s.endswith('_volume')]
    for org_vol in _vol:
        area_fraction=[]
        for idx in regions_summary.index.unique():
            ov = regions_summary.loc[idx][(org_vol, 'sum')]
            rv = regions_summary.loc[idx][('volume', 'sum')]
            afrac = ov/rv
            area_fraction.append(afrac)
        regions_summary[(org_vol, 'fraction')] = area_fraction
    return regions_summary

In [13]:
def summarize_regions(regions_df, 
                      org_list: list[str],
                      group_by:list[str]=['dataset', 'image_name', 'object', 'cell'], 
                      shared_columns:list[str]=["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"],
                      ag_func_standard: list[str]=['mean', 'median', 'std'],
                      zeros=None):
    tab1 = regions_df[group_by + ['volume']].groupby(group_by).agg(['count', 'sum'] + ag_func_standard)
    tab2 = regions_df[group_by + ['surface_area'] + [f'{org}_volume' for org in org_list]].groupby(group_by).agg(['sum'] + ag_func_standard)
    tab3 = regions_df[group_by + shared_columns].groupby(group_by).agg(ag_func_standard)
    shared_metrics = pd.merge(tab1, tab2, 'outer', on=group_by)
    regions_summary = pd.merge(shared_metrics, tab3, 'outer', on=group_by)
    count_one = [count==1 for count in regions_summary["volume"]["count"]] # List of bools for each row having 1 as its count or not
    regions_summary.loc[count_one, (["volume"],["mean"])] = zeros
    regions_summary.loc[count_one, (["surface_area"],["mean"])] = zeros
    for col in shared_columns+["volume", "surface_area"]:
        regions_summary.loc[count_one, ([col],["median"])] = zeros
        regions_summary.loc[count_one, ([col],["std"])] = zeros

    return regions_summary

In [14]:
def finalize_regions(regions_summary):
    regions_final = regions_summary.unstack(-1)
    regions_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in regions_final.columns.to_flat_index()]
    regions_final['nuc_area_fraction'] = regions_final['nuc_mean_volume'] / (regions_final['cell_mean_volume'])
    regions_final = regions_final.reset_index()
    for col in regions_final.columns:
        if col.endswith(("_count_volume","_sum_volume", "_mean_volume", "_median_volume")):
            regions_final[col] = regions_final[col].fillna(0)
        if col.endswith("_count_volume"):
            regions_final.rename(columns={col:col.split("_")[0]+"_count"}, inplace=True)
    regions_final = regions_final.reset_index()
    return regions_final

In [15]:
def zeros_singles(input_df: pd.DataFrame, shared_columns: list[str], zeros = None):
    for region in input_df.columns.get_level_values(0).unique():
        count_one = [count==1 for count in input_df[region]["volume"]["count"]] # List of bools for each row having 1 as its count or not
        input_df.loc[count_one, ([region],["volume"],["mean"],)] = zeros
        input_df.loc[count_one, ([region],["surface_area"],["mean"])] = zeros
        for col in shared_columns+["volume", "surface_area"]:
            input_df.loc[count_one, ([region],[col],["median"])] = zeros
            input_df.loc[count_one, ([region],[col],["std"])] = zeros
    return input_df

In [16]:
def subregion_subunit_combinder(in_df):
    def border_type_combinder(x):
        return x.split(':')[0]+':'+"_".join(list(set([i.split('-')[0] for i in x.split(':')[1].split('_')])))
    out_df = in_df.copy()
    is_border = out_df['subregion'].str.split(':').str[0] == 'border'
    out_df.loc[is_border, 'subregion'] = out_df.loc[is_border, 'subregion'].apply(lambda x: border_type_combinder(x))
    
    not_border = [not val for val in is_border]
    out_df.loc[not_border, 'subregion'] = out_df.loc[not_border, 'subregion'].str.split('-').str[0]
    out_df['subregion_separated'] = in_df['subregion']
    return out_df

## Distribution

In [17]:
def summarize_dist(dist_df, group_by): #Look into colapsing data into one fn

    sel = group_by.copy()
    sel.remove('object')

    #dist_df = dist_df.set_index(group_by+['subregion']).unstack('subregion').swaplevel(i=0, j=-1, axis=1)
    
    hist_dfs = []
    for ind in dist_df.index:
        selection = dist_df.loc[[ind]]
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()

        bins_df[['bins', 'masks', 'obj']] = selection[['XY_bins', 'XY_mask_vox_cnt_perbin', 'XY_obj_vox_cnt_perbin']]
        wedges_df[['bins', 'masks', 'obj']] = selection[['XY_wedges', 'XY_mask_vox_cnt_perwedge', 'XY_obj_vox_cnt_perwedge']]
        Z_df[['bins', 'masks', 'obj']] = selection[['Z_slices', 'Z_mask_vox_cnt', 'Z_obj_vox_cnt']]

        dfs = [selection[group_by].reset_index()]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["obj"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "))), columns =['bins', 'obj', 'mask']).astype(int)
            
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            single_df['obj_norm'] = (single_df["obj"]/single_df["mask_fract"]).fillna(0)
            single_df['portion_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100

            if "Z_" in prefix:
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*10).apply(np.floor)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_norm'])
            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_norm'].sum() != 0: sumstats_df['hist_mode']=[s.mode()[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())
        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        hist_dfs.append(combined_df)
    dist_org_summary = pd.concat(hist_dfs, ignore_index=True)

    # nucleus distribution
    nuc_dist_df = dist_df[sel + ["XY_bins", "XY_center_vox_cnt_perbin", "XY_mask_vox_cnt_perbin",
                                 "XY_wedges", "XY_center_vox_cnt_perwedge", "XY_mask_vox_cnt_perwedge",
                                 "Z_slices", "Z_center_vox_cnt", "Z_mask_vox_cnt"]].set_index(sel)
    nuc_hist_dfs = []
    for idx in nuc_dist_df.index.unique():
        selection = nuc_dist_df.loc[idx].iloc[[0]].reset_index()
        bins_df = pd.DataFrame()
        wedges_df = pd.DataFrame()
        Z_df = pd.DataFrame()

        bins_df[['bins', 'center', 'masks']] = selection[['XY_bins', 'XY_center_vox_cnt_perbin', 'XY_mask_vox_cnt_perbin']]
        wedges_df[['bins', 'center', 'masks']] = selection[['XY_wedges', 'XY_center_vox_cnt_perwedge', 'XY_mask_vox_cnt_perwedge']]
        Z_df[['bins', 'center', 'masks']] = selection[['Z_slices', 'Z_center_vox_cnt', 'Z_mask_vox_cnt']]

        dfs = [selection[sel]]
        for df, prefix in zip([bins_df, wedges_df, Z_df], ["XY_bins_", "XY_wedges_", "Z_slices_"]):
            single_df = pd.DataFrame(list(zip(df["bins"].values[0][1:-1].split(", "), 
                                            df["masks"].values[0][1:-1].split(", "),
                                            df["center"].values[0][1:-1].split(", "))), columns =['bins', 'mask', 'obj']).astype(int)
            single_df['mask_fract'] = single_df['mask']/single_df['mask'].max()
            single_df['obj_norm'] = (single_df["obj"]/single_df["mask_fract"]).fillna(0)
            single_df['portion_per_bin'] = (single_df["obj"] / single_df["obj"].sum())*100
            if "Z_" in prefix:
                single_df['bins'] = (single_df["bins"]/max(single_df.bins)*10).apply(np.floor)

            sumstats_df = pd.DataFrame()

            s = single_df['bins'].repeat(single_df['obj_norm'])
            sumstats_df['hist_mean']=[s.mean()]
            sumstats_df['hist_median']=[s.median()]
            if single_df['obj_norm'].sum() != 0: sumstats_df['hist_mode']=[s.mode()[0]]
            else: sumstats_df['hist_mode']=['NaN']
            sumstats_df['hist_min']=[s.min()]
            sumstats_df['hist_max']=[s.max()]
            sumstats_df['hist_range']=[s.max() - s.min()]
            sumstats_df['hist_stdev']=[s.std()]
            sumstats_df['hist_skew']=[s.skew()]
            sumstats_df['hist_kurtosis']=[s.kurtosis()]
            sumstats_df['hist_var']=[s.var()]
            sumstats_df.columns = [prefix+col for col in sumstats_df.columns]
            dfs.append(sumstats_df.reset_index())
        combined_df = pd.concat(dfs, axis=1).drop(columns="index")
        nuc_hist_dfs.append(combined_df)
    dist_center_summary = pd.concat(nuc_hist_dfs, ignore_index=True)
    dist_center_summary.insert(3, column="object", value="nuc")

    ddf = pd.concat([dist_org_summary, dist_center_summary], axis=0).set_index(group_by).unstack('subregion').swaplevel(i=0, j=-1, axis=1)
    ddf = ddf.reindex(ddf.columns.get_level_values(0).unique(), level=0, axis=1)

    return ddf.sort_index()

# New OVERALL Function

In [18]:
def batch_summary_stats(csv_path_list: List[str],
                         out_path: str,
                         out_preffix: str,
                         splitter:str="X"):
    """" 
    csv_path_list: List[str],
        A list of path strings where .csv files to analyze are located.
    out_path: str,
        A path string where the summary data file will be output to
    out_preffix: str
        The prefix used to name the output file.    
    """
    ds_count = 0
    fl_count = 0
    ###################
    # Read in the csv files and combine them into one of each type
    ###################
    org_df, interaction_df, dist_df, regions_df, ds_count, fl_count = file_reader(csv_path_list=csv_path_list)

    cell_number = org_df.columns[org_df.columns.get_loc('object')+1]

    org_df = subregion_subunit_combinder(org_df)
    interaction_df = subregion_subunit_combinder(interaction_df)
    dist_df = subregion_subunit_combinder(dist_df)
    
    org_df['cell_col'] =  org_df[cell_number].str.split('-').str[0] 
    interaction_df['cell_col'] = interaction_df[cell_number].str.split('-').str[0]
    
    ###################
    # adding new metrics to the original sheets
    ###################
    # TODO: include these labels when creating the original sheets
    interaction_cnt = batch_summary_interactions(interaction_df=interaction_df, splitter=splitter)
    org_df = pd.merge(org_df, interaction_cnt, how='left', on=['dataset', 'image_name', cell_number, 'object', 'label'], sort=True)
    org_df[interaction_cnt.columns] = org_df[interaction_cnt.columns].fillna(0)

    ###################
    # summary stat group
    ###################
    group_by = ['dataset', 'image_name', cell_number, 'object']
    sharedcolumns = ["SA_to_volume_ratio", "equivalent_diameter", "extent", "euler_number", "solidity", "axis_major_length"]
    ag_func_standard = ['mean', 'median', 'std']

    ###################
    # group metrics from regions_df
    ###################
    regions_summary = summarize_regions(regions_df=regions_df, org_list=org_df.object.unique(), group_by=group_by, shared_columns=sharedcolumns)
    
    ###################
    # find regions that need multiple instances labeled
    ###################
    multi_regions = [idx for idx in regions_summary.index if regions_summary.loc[idx]["volume", "count"]>1]
    multi_regions_interaction = [tuple(row) in multi_regions for idx, row in interaction_df[group_by[:-1]+['subregion']].iterrows()]
    multi_regions_org = [tuple(row) in multi_regions for idx, row in org_df[group_by[:-1]+['subregion']].iterrows()]
    border_regions_interaction = ['border:' in str(tuple(row)[-1]) for idx, row in interaction_df[group_by + ['subregion_separated']].iterrows()]
    border_regions_org = ['border:' in str(tuple(row)[-1]) for idx, row in org_df[group_by + ['subregion_separated']].iterrows()]
    multi_border_regions_interaction = [a or b for a, b in zip(multi_regions_interaction, border_regions_interaction)]
    multi_border_regions_org = [a or b for a, b in zip(multi_regions_org, border_regions_org)]

    ###################
    # summarize shared measurements between org_df and contacts_df
    ###################
    org_summary = org_summarize(org_df, group_by=group_by, shared_columns=sharedcolumns, ag_func_standard=ag_func_standard, multi_regions=multi_border_regions_org)
    interaction_summary = interaction_summarize(interaction_df, group_by=group_by, shared_columns=sharedcolumns, ag_func_standard=ag_func_standard, multi_regions=multi_border_regions_interaction)

    # set single metrics to zeros (no median or standard deviation)
    org_summary = zeros_singles(org_summary, sharedcolumns)
    interaction_summary = zeros_singles(interaction_summary, sharedcolumns)

    columns2 = [col for col in org_df.columns if col.endswith(("_count", "_volume"))]
    contact_counts_summary_subcell = org_df[group_by + ['subregion'] + columns2].groupby(group_by+['subregion']).agg(['sum'] + ag_func_standard).unstack('subregion').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    contact_counts_summary_cell = org_df[group_by + ['cell_col'] + columns2].groupby(group_by+['cell_col']).agg(['sum'] + ag_func_standard).unstack('cell_col').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)
    contact_counts_summary_subcell_counts = org_df.loc[multi_border_regions_org, group_by + ['subregion_separated'] + columns2].groupby(group_by+['subregion_separated']).agg(['sum'] + ag_func_standard).unstack('subregion_separated').swaplevel(i=0, j=-1, axis=1).swaplevel(i=1, j=2, axis=1)

    org_summary = pd.merge(org_summary, contact_counts_summary_subcell, 'outer', on=group_by)
    org_summary = pd.merge(org_summary, contact_counts_summary_cell, 'outer', on=group_by)
    org_summary = pd.merge(org_summary, contact_counts_summary_subcell_counts, 'outer', on=group_by)

    ###################
    # summarize distribution measurements
    ###################
    # organelle distributions
    dist_summary = summarize_dist(dist_df, group_by+['subregion'])
    ###################
    # add normalization
    ###################
    # organelle area fraction

    org_summary = org_area_fraction(org_summary, regions_summary, regions_df, group_by)
    # TODO: add in line to reorder the level=0 columns here

    # regions area fraction
    regions_summary = region_volume_fraction(regions_summary=regions_summary)
    
    # contact sites volume normalized
    interaction_summary = normalize_interaction_volumes(interaction_summary, regions_summary, regions_df, group_by, splitter)

    # number and area of individuals organelle involved in contact
    org_summary = normalize_org(org_df, org_summary, group_by, multi_border_regions_org)

    # reorder columns 
    org_summary = org_summary.reindex(org_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    org_summary = org_summary.reindex(org_summary.columns.get_level_values(1).unique(), level=1, axis=1)
    
    interaction_summary = interaction_summary.reindex(interaction_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    interaction_summary = interaction_summary.reindex(interaction_summary.columns.get_level_values(1).unique(), level=1, axis=1)
    
    dist_summary = dist_summary.reindex(dist_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    
    regions_summary = regions_summary.reindex(regions_summary.columns.get_level_values(0).unique(), level=0, axis=1)
    

    ###################
    # flatten datasheets and combine
    # TODO: restructure this so that all of the datasheets and unstacked and then reorded based on shared level 0 columns before flattening
    ###################
    # org flattening
    org_final = org_flattening(org_summary, org_df, splitter)

    interaction_final = interaction_flattening(interaction_summary)

    # distributions flattened
    dist_final = dist_summary.unstack(-1)
    dist_final.columns = ["_".join((col_name[-1], col_name[1], col_name[0])) for col_name in dist_final.columns.to_flat_index()]
    dist_final = dist_final.reset_index()

    # regions flattened & normalization added
    regions_final=finalize_regions(regions_summary)
    # combining them all
    combined = pd.merge(regions_final, dist_final, on=["dataset", "image_name", cell_number], how="outer")
    combined = pd.merge(interaction_final, combined, on=["dataset", "image_name", cell_number], how="outer")
    combined = pd.merge(org_final, combined, on=["dataset", "image_name", cell_number], how="outer").set_index(["dataset", "image_name", cell_number])
    combined.columns = [col.replace('sum', 'total') for col in combined.columns]

    ###################
    # export summary sheets
    ###################
    org_summary.to_csv(out_path + f"/{out_preffix}per_org_summarystats.csv")
    interaction_summary.to_csv(out_path + f"/{out_preffix}per_contact_summarystats.csv")
    dist_summary.to_csv(out_path + f"/{out_preffix}distribution_summarystats.csv")
    regions_summary.to_csv(out_path + f"/{out_preffix}per_region_summarystats.csv")
    combined.to_csv(out_path + f"/{out_preffix}summarystats_combined.csv")

    print(f"Processing of {fl_count} files from {ds_count} dataset(s) is complete.")
    return f"{fl_count} files from {ds_count} dataset(s) were processed"

In [19]:
batch_summary_stats([Path("C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/raquel/out")],
                    "C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/raquel/out/summary",
                    out_preffix="neurite_soma_sumstat")

C:\Users\zscoman\AppData\Local\Temp\ipykernel_20908\3172471891.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interaction_cnt[[f"org{cha}" for cha in string.ascii_uppercase[:(len(max(interaction_cnt["object"].str.split(splitter), key=len)))]]] = interaction_cnt["object"].str.split(splitter, expand=True)
C:\Users\zscoman\AppData\Local\Temp\ipykernel_20908\3172471891.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  interaction_cnt[[f"org{cha}" for cha in string.ascii_uppercase[:(len(max(interaction_

KeyError: ('neurite_checks_neurites_soma', 'C2C12_20_CMA_new_settings_zstack_10_Linear_unmixing_0_cmle.ome', 'cell-1', 'x')